The target columns we need to join this on the WCV data follow:

- [x] OrganizationName
- [x] CaseNumber - Will need to be calculated - {Year}-{admission ID}
- [x] PatientID
- [x] CommonSpeciesName
- [x] GeneralSpeciesName 
- [] DVertebrate - (Bird, Mammal, Reptile, Amphibian)
- [] DateAdmitted
- [] DDateAdmittedYear
- [] Season
- [] DDateAdmittedMonth
- [] DDateAdmittedDOM
- [] DDateAdmittedDOW
- [] CircumstancesOfRescue - Remove
- [] RescueState - Remove
- [] RescueJurisdiction - Remove
- [] RescueAddress - Remove
- [] OtherRescueInformation - Remove
- [] Latitude
- [] Longitude
- [] Elevation - Remove
- [] Disposition - This is the Original 
- [] UpdatedDisposition - Unified language (Active, Died, Released, Transferred)
- [] DayOfWeekNumber 
- [] MonthNumber

TODO:
- [ ] Check with Linda about the organization name - is there a way to query it from WRMD
- [ ] See about how to get unique identifiers for each record in WRMD
- [ ] Double check the common names against the existing wcv common names for consistency
- [ ] Use the CollisionAnimalMapping CSV to create the General Species Name column, check if all common species exist in it or we need to add more mappings


In [2]:
import pandas as pd

In [3]:
# Load in pickled dataframe
df = pd.read_pickle("./datasets/WRMD_2016_to_2024_all_cols_geocoding_not_req.pkl")

In [4]:
df

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
0,2016,NaN,1,0.0,Adult,Depressed,Good,NaN,congested breathing,"unable to stay sternal, ataxia",...,--,--,508-360-2061,22931,VA,Mammalia,Didelphidae,Didelphis,Didelphimorphia,virginiana
2,2016,NaN,3,0.0,Adult,Alert,Emaciated,NaN,NaN,NaN,...,brought in by Erin,--,304-876-1068,--,OH,Aves,Cathartidae,Coragyps,Cathartiformes,atratus
8,2016,NaN,9,0.0,Adult,Depressed,Thin,NaN,NaN,barely responsive,...,--,--,5403032556,22603,VA,Aves,Strigidae,Strix,Strigiformes,varia
10,2016,NaN,11,0.0,Adult,Quiet,Thin,NaN,NaN,NaN,...,willing to release when ready,Frederick County Landfill,3046715230,--,VA,Aves,Strigidae,Strix,Strigiformes,varia
16,2016,NaN,17,0.0,Adult,Quiet,Reasonable,NaN,rapid and shallow breathing,NaN,...,--,--,7037370110,20176,VA,Aves,Turdidae,Turdus,Passeriformes,migratorius
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24955,2024,NaN,3752,NaN,Adult,Obtunded,Good,NaN,"moderately increased respiratory effort, open ...",laterally recumbent in finder's backseat,...,NaN,NaN,9154870726,25430,WV,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
24996,2024,NaN,3793,NaN,Juvenile,Quiet,Reasonable,NaN,NaN,NaN,...,NaN,NaN,2672661277,18360,PA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25024,2024,NaN,3821,NaN,Adult,Alert,Reasonable,NaN,NaN,weak leg use,...,NaN,NaN,5408776642,22602,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25049,2024,NaN,3846,NaN,Juvenile,Quiet,Plump,NaN,NaN,NaN,...,NaN,Stafford County Animal Control,NaN,NaN,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus


In [5]:
# Add a column to the dataframe named "OrganizationName" and fill it with the value "WRMD"
df["OrganizationName"] = "WRMD"


In [6]:
# Create a new columns called "Case Number" where the value is based on the "case year" and "id" columns
df["Case Number"] = df["admissions.case_year"].astype(str) + "-" + df["admissions.id"].astype(str)

In [7]:
df["Case Number"]


0           2016-1
2           2016-3
8           2016-9
10         2016-11
16         2016-17
           ...    
24955    2024-3752
24996    2024-3793
25024    2024-3821
25049    2024-3846
25058    2024-3855
Name: Case Number, Length: 1113, dtype: object

In [8]:
# Create a column called PatientID where the value is empty
df["PatientID"] = None

In [9]:
# Create a columnn called CommonSpeciesName based on the patient.common_name column
df["CommonSpeciesName"] = df["patients.common_name"].str.lower()

In [30]:
# We need to check that every unique value in the CommonSpeciesName column is in the CollisionAnimalMapping.csv file

collision_mapping = pd.read_csv("./CollisionAnimalMapping.csv")
# Check if all unique values in CommonSpeciesName are in the collision_mapping
unique_common_names = df["CommonSpeciesName"].unique()
missing_names = set(unique_common_names) - set(collision_mapping["Animal"].str.lower())
assert len(missing_names) == 0, f"Missing common names in collision mapping: {missing_names}"

In [31]:
# Create a column called GeneralSpeciesName based on looking up the CommonSpeciesName in the CollisionAnimalMapping.csv file
collision_mapping["Animal"] = collision_mapping["Animal"].str.lower()
df["GeneralSpeciesName"] = df["CommonSpeciesName"].map(
    dict(zip(collision_mapping["Animal"], collision_mapping["Mapping"]))
)

In [34]:
df[['GeneralSpeciesName', 'CommonSpeciesName']]
# assert that if CommonSpeciesName is not null, GeneralSpeciesName is not null
assert df[df["CommonSpeciesName"].notnull()]["GeneralSpeciesName"].notnull().all(), "Some CommonSpeciesName values are missing a GeneralSpeciesName"